# Correlación entre representation metrics y `test_acc`

Este notebook toma el `.pkl` generado por `parse_out_repr_metrics.py`, extrae `test_acc` por época desde los `output.log` de cada run y calcula correlaciones (Pearson y Spearman) entre cada métrica de representación y `test_acc`.


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)

FLOAT_PATTERN = r"([-+]?\d*\.?\d+(?:[eE][-+]?\d+)?)"


In [ ]:
# === Configuración ===
REPR_PKL_PATH = Path("cars3d_repr_epochs.pkl")
# Si quieres usar otro dataset, cambia la ruta arriba.

# Si True, también calcula correlaciones agregando por run (promedio temporal de cada métrica y mejor test_acc por run).
CALC_RUN_LEVEL = True

assert REPR_PKL_PATH.exists(), f"No existe: {REPR_PKL_PATH.resolve()}"


In [ ]:
df_repr = pd.read_pickle(REPR_PKL_PATH)
print(f"Filas: {len(df_repr):,}")
print(df_repr.columns.tolist())
df_repr.head()


In [ ]:
required_cols = {"run_path", "epoch"}
missing = required_cols - set(df_repr.columns)
assert not missing, f"Faltan columnas requeridas en el PKL: {missing}"

repr_metric_cols = [
    c for c in df_repr.columns
    if c.startswith("val_4cases_")
]
print("Métricas de representación detectadas:")
repr_metric_cols


In [ ]:
def find_wandb_log_path(run_path: Path) -> Path | None:
    candidates = [
        run_path / "wandb" / "latest-run" / "files" / "output.log",
        run_path / "output.log",
        run_path / "wandb" / "output.log",
    ]
    for c in candidates:
        if c.exists():
            return c

    wandb_root = run_path / "wandb"
    roots = [wandb_root] if wandb_root.is_dir() else [run_path]

    found = []
    for root in roots:
        for p in root.rglob("output.log"):
            found.append(p)
    if not found:
        return None
    return max(found, key=lambda p: p.stat().st_mtime)


def extract_test_acc_by_epoch(log_data: str) -> dict[int, float]:
    epoch_data = {}
    epoch_pattern = re.compile(r"Epoch \[(\d+)\]")
    chunks = epoch_pattern.split(log_data)[1:]

    for i in range(0, len(chunks), 2):
        epoch = int(chunks[i].strip())
        content = chunks[i + 1]
        parsed = re.search(rf"test_acc:\s*{FLOAT_PATTERN}", content)
        epoch_data[epoch] = float(parsed.group(1)) if parsed else np.nan
    return epoch_data


def build_test_acc_df(run_paths: pd.Series) -> pd.DataFrame:
    rows = []
    for run in sorted(set(run_paths.dropna().astype(str))):
        run_path = Path(run)
        log_path = find_wandb_log_path(run_path)
        if log_path is None:
            continue
        try:
            text = log_path.read_text()
        except Exception:
            continue

        by_epoch = extract_test_acc_by_epoch(text)
        for epoch, test_acc in by_epoch.items():
            rows.append({"run_path": run, "epoch": epoch, "test_acc": test_acc})

    return pd.DataFrame(rows)


In [ ]:
df_test = build_test_acc_df(df_repr["run_path"])
print(f"Filas de test_acc extraídas: {len(df_test):,}")
df_test.head()


In [ ]:
df = df_repr.merge(df_test, on=["run_path", "epoch"], how="left")
print(df[["run_path", "epoch", "test_acc"]].isna().mean())
print(f"Filas combinadas: {len(df):,}")
df.head()


In [ ]:
def corr_table(data: pd.DataFrame, metrics: list[str], target: str = "test_acc") -> pd.DataFrame:
    rows = []
    for m in metrics:
        sub = data[[m, target]].dropna()
        n = len(sub)
        if n < 3:
            rows.append({"metric": m, "n": n, "pearson": np.nan, "spearman": np.nan})
            continue
        rows.append({
            "metric": m,
            "n": n,
            "pearson": sub[m].corr(sub[target], method="pearson"),
            "spearman": sub[m].corr(sub[target], method="spearman"),
        })
    out = pd.DataFrame(rows)
    return out.sort_values("spearman", ascending=False)


corr_epoch = corr_table(df, repr_metric_cols, target="test_acc")
corr_epoch


In [ ]:
# Heatmap de correlaciones por época
plt.figure(figsize=(6, max(4, 0.45 * len(corr_epoch))))
heat_data = corr_epoch.set_index("metric")[["pearson", "spearman"]]
sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="coolwarm", center=0)
plt.title("Correlación con test_acc (nivel época)")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter para las 3 métricas con mayor |Spearman|
top3 = corr_epoch.assign(abs_s=lambda x: x["spearman"].abs()).sort_values("abs_s", ascending=False).head(3)["metric"].tolist()

fig, axes = plt.subplots(1, len(top3), figsize=(6 * len(top3), 4), squeeze=False)
for ax, metric in zip(axes[0], top3):
    sns.regplot(data=df, x=metric, y="test_acc", scatter_kws={"alpha": 0.25, "s": 15}, line_kws={"color": "crimson"}, ax=ax)
    ax.set_title(metric)
plt.tight_layout()
plt.show()


In [ ]:
if CALC_RUN_LEVEL:
    # Agregación por run: promedio temporal de métricas de representación y mejor test_acc por run.
    agg_dict = {m: "mean" for m in repr_metric_cols}
    agg_dict["test_acc"] = "max"

    run_level = (
        df[["run_path", *repr_metric_cols, "test_acc"]]
        .groupby("run_path", as_index=False)
        .agg(agg_dict)
    )

    corr_run = corr_table(run_level, repr_metric_cols, target="test_acc")
    display(corr_run)

    plt.figure(figsize=(6, max(4, 0.45 * len(corr_run))))
    heat_data = corr_run.set_index("metric")[["pearson", "spearman"]]
    sns.heatmap(heat_data, annot=True, fmt=".3f", cmap="coolwarm", center=0)
    plt.title("Correlación con test_acc (nivel run)")
    plt.tight_layout()
    plt.show()


## Notas

- Si ya tienes `test_acc` dentro del `.pkl`, puedes saltarte la extracción desde logs y usar esa columna directamente.
- Si prefieres otra estrategia de agregación por run (por ejemplo, métrica en la *best epoch* por `val_acc`), se puede adaptar en una celda.
- Recomendación: reportar **Pearson y Spearman** porque muchas relaciones entre métricas de representación y accuracy no son lineales.
